In [0]:
CATALOG = "energy_puls"
SILVER = f"{CATALOG}.silver"
GOLD   = f"{CATALOG}.gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.renovation_priority AS
SELECT
    c.code_commune,
    c.nom_commune,
    c.population,
    COUNT(d.numero_dpe)                                    AS nb_logements,
    ROUND(AVG(d.conso_kwh_m2_an), 1)                       AS conso_moyenne,
    ROUND(100.0 * SUM(CASE WHEN d.classe_dpe IN ('F','G')
                           THEN 1 ELSE 0 END)
          / COUNT(*), 1)                                   AS pct_passoires
FROM {SILVER}.dpe_clean d
JOIN {SILVER}.communes c ON d.code_commune = c.code_commune
GROUP BY 1,2,3
HAVING COUNT(d.numero_dpe) >= 20
ORDER BY pct_passoires DESC
""")

display(spark.sql(f"SELECT * FROM {GOLD}.renovation_priority LIMIT 20"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.conso_vs_temperature AS
SELECT
    m.region_name,
    ROUND(m.temperature)              AS temp_arrondie,
    COUNT(*)                          AS nb_heures,
    ROUND(AVG(e.consommation), 0)     AS conso_moyenne_mw
FROM {SILVER}.meteo_clean m
JOIN {CATALOG}.bronze.eco2mix_raw e
  ON e.libelle_region = m.region_name
 AND date_format(e.date_heure, 'yyyy-MM-dd HH') = date_format(m.heure, 'yyyy-MM-dd HH')
GROUP BY 1,2
ORDER BY region_name, temp_arrondie
""")

display(spark.sql(f"""
  SELECT * FROM {GOLD}.conso_vs_temperature
  WHERE region_name = 'Bretagne' ORDER BY temp_arrondie
"""))